# Khai phá Dữ liệu Tuyển sinh Đại học Việt Nam (Data Mining)
## Notebook 08: Dự báo Điểm chuẩn Tuyển sinh

Notebook này thực hiện:
1. Phân tích chuỗi thời gian điểm chuẩn của một số trường/ngành học tiêu biểu.
2. Huấn luyện mô hình hồi quy tuyến tính (Linear Regression) để dự báo xu hướng ngắn hạn.
3. Huấn luyện mô hình ARIMA để dự báo các chuỗi thời gian có biến động.
4. Đánh giá sai số dự báo của mô hình (MAE, RMSE).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.mining.forecasting import ScoreForecaster

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style="whitegrid")

df = pd.read_csv("../data/processed/admission_processed.csv", encoding="utf-8-sig")

### 1. Khởi tạo bộ dự báo và xem dữ liệu chuỗi thời gian mẫu

In [2]:
forecaster = ScoreForecaster(df)
# Dự báo thử điểm chuẩn ngành Công nghệ thông tin của Đại học Bách khoa Hà Nội năm 2026
res = forecaster.forecast_school_major(
    school_name="Đại học Bách khoa Hà Nội",
    major_name="Công nghệ thông tin",
    forecast_year=2026
)
print("Kết quả dự báo chi tiết:")
for k, v in res.items():
    print(f"  {k:<20}: {v}")

Kết quả dự báo chi tiết:
  history             : {np.int64(2020): np.float64(28.204117647058826), np.int64(2021): np.float64(28.50294117647059), np.int64(2022): np.float64(27.774705882352944), np.int64(2023): np.float64(27.963529411764707), np.int64(2024): np.float64(28.105882352941176), np.int64(2025): np.float64(28.123529411764704)}
  predicted_score     : 27.97
  lower_bound         : 27.56
  upper_bound         : 28.39
  model_used          : LinearRegression


/home/vietpv/miniconda3/envs/testing/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/home/vietpv/miniconda3/envs/testing/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


### 2. Vẽ biểu đồ Lịch sử điểm chuẩn và Kết quả dự báo năm 2026

In [3]:
history = res['history']
years = list(history.keys()) + [2026]
scores = list(history.values()) + [res['predicted_score']]

plt.figure(figsize=(10, 5))
plt.plot(list(history.keys()), list(history.values()), marker='o', label='Thực tế (2020-2025)', color='blue', linewidth=2)
plt.plot([2025, 2026], [list(history.values())[-1], res['predicted_score']], marker='d', linestyle='--', label='Dự báo (2026)', color='red', linewidth=2)
plt.title('Biểu đồ Lịch sử & Dự báo Điểm chuẩn Ngành CNTT - Bách khoa Hà Nội')
plt.xlabel('Năm')
plt.ylabel('Điểm chuẩn')
plt.xticks(years)
plt.legend()
plt.show()

/tmp/ipykernel_58906/656715547.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3. Đánh giá chất lượng mô hình dự báo tổng thể

In [4]:
metrics = forecaster.evaluate_forecasting_models()
print("Chỉ số đánh giá độ chính xác của các mô hình dự báo:")
for k, v in metrics.items():
    print(f"  {k:<10}: {v:.4f}")

2026-06-29 17:13:24.201 | INFO     | src.mining.forecasting:evaluate_model:279 - Evaluation: MAE=0.646, RMSE=0.727, MAPE=3.27%


Chỉ số đánh giá độ chính xác của các mô hình dự báo:
  MAE       : 0.6460
  RMSE      : 0.7270
  MAPE      : 3.2700
